Functions needed to precondition CG with preconditioner as described in "Fast Iteratively Reweighted Least Squares
Algorithms for Analysis-Based Sparsity
Reconstruction"


Function that creates a, b, c as in Figure (41)
Function that solves lower triangular system with input the diagonals
Function that solves upper triangular system with input the diagonals

In [7]:
import numpy as np
import math
import scipy as sp
from cil.optimisation.functions import L2NormSquared

Computing $\overline{A^TA}$

In [8]:
def mean_ATA(A, n_pixels):
    """
    A is a ProjectionOperator
    """
    two_norm = L2NormSquared()
    get_ig = A.domain_geometry()

    sum = 0
    for k in range(3):

        # generate random entries
        xk = np.random.normal(0, 1, (n_pixels, n_pixels))

        # put random entries in class <DataContainer>
        xk_datacon = get_ig.allocate()
        xk_datacon.fill(xk)

        # compute 
        yk = A.direct(xk_datacon)
        sum += two_norm(yk)
    
    final = (1/3)*sum
    return final

solving LU 

In [9]:
def create_abs(A, weights, n):
    """
    weights in np array from np.shape(weights) = n_pixels X n_pixels

    n: number of pixels
    """

    a_1 = weights.flatten()
    a_2 = weights.flatten()

    pix_squared = len(a_1)
    for i in range(pix_squared - 1):
        a_1[i] += a_1[i+1]

    for i in range(pix_squared - n):
        a_2[i] += a_2[i + n]
    
    ATAI_mean = mean_ATA(A, n)

    a = a_1 + a_2 + ATAI_mean

    b = (-weights.flatten())[1:]
    c = (-weights.flatten())[n:]
    return a, b, c

def solve_Ux(diag, updiag, offdiag, y):
    """
   diag has length N, then off diag has lenth N-n and is put appropiatly in the matrix system

    Solves Ux = y
    """
    N_big = len(diag)
    n = int(np.sqrt(N_big))

    x = np.zeros(N_big)
    x[-1] = y[-1] / diag[-1]    # setting x[N_big - 1]

    for i in range(2, N_big + 1):   # i loops 2, 3, 4, 5, 6
        k = N_big - i
        if k >= N_big - n:   # N_big - i loops 
            print("first if", N_big - i)
            x[k] = (1 / diag[k]) * (y[k] - updiag[k]*x[k + 1])
        if k < N_big - n:
            print("second if", N_big - i)
            x[k] = (1 / diag[k]) * (y[k] - updiag[k]*x[k + 1] - offdiag[k] * x[k + n])
    return x

def solve_Lx(lowdiag, offdiag, y):
    
    N_big = len(y)
    n = int(np.sqrt(N_big))
    x = np.zeros(N_big)

    x[0] = y[0]
    for i in range(1, N_big):
        if i < n:
            x[i] = y[i] - lowdiag[i - 1]*x[i - 1]
        if i >= n:
            x[i] = y[i] - lowdiag[i - 1]*x[i - 1] - offdiag[i - n]*x[i - n]

    return x


In [12]:
def solve_Pxy(P_a, P_b, P_c, y):
    
    """
    solves Px = y

    P_a, P_b, P_c are defined as in the article figure (41)

    x: returned in np array form
    y: ImageData
    """

    L_lowdiag = P_b[:-1] / P_a[:-1]
    L_offdiag = P_c / P_a[:len(P_c)]

    y_array = y.as_array

    temp_solve = solve_Lx(L_lowdiag, L_offdiag, y_array)
    x = solve_Ux(P_a, P_b, P_c, temp_solve)

    return x